# DL GenAI Project: Milestone 1 baseline
**Learner:** Sripathi Dhruv Reddy (`23f3001736`)  
**Competition:** SurfaceGuard AI: Industrial Surface Defect Detection

This notebook follows the Milestone 1 guideline: combination-stratified 80/20
split, a native-resolution segmentation baseline trained for five epochs, a
threshold sweep at 0.3/0.5/0.7/0.9 without area filtering, and a real test
submission generated from the trained checkpoint.

In [ ]:
from pathlib import Path
import gc, json, os, random

import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 1736
H, W = 128, 800
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
THRESHOLDS = [0.3, 0.5, 0.7, 0.9]
EPOCHS = 5
BATCH_SIZE = 16

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## Locate the official competition files

In [ ]:
roots = list(Path("/kaggle/input").glob("**/public/train.csv"))
if not roots:
    roots = list(Path("/kaggle/input").glob("**/train.csv"))
assert roots, "Official train.csv was not found. Attach the competition data source."
TRAIN_CSV = roots[0]
DATA_ROOT = TRAIN_CSV.parent
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
SAMPLE_CSV = DATA_ROOT / "sample_submission.csv"
print({"train_csv": str(TRAIN_CSV), "train_dir": str(TRAIN_DIR), "test_dir": str(TEST_DIR)})

train_long = pd.read_csv(TRAIN_CSV, keep_default_na=False)
sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
assert list(train_long.columns) == ["image_id", "defect_type", "mask_rle", "width", "height"]
assert list(sample.columns) == ["ID", "image_id", "defect_type", "mask_rle"]
print("train rows/images:", len(train_long), train_long.image_id.nunique())
print("test rows/images:", len(sample), sample.image_id.nunique())

## RLE and combination-stratified 80/20 split

Competition RLE uses 1-based starts and column-major (Fortran) flattening.

In [ ]:
def rle_decode(rle, shape=(H, W)):
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    text = str(rle).strip()
    if text:
        values = np.asarray(text.split(), dtype=np.int64).reshape(-1, 2)
        for start, length in values:
            mask[start - 1:start - 1 + length] = 1
    return mask.reshape(shape, order="F")


def rle_encode(mask):
    pixels = np.asarray(mask, dtype=np.uint8).reshape(-1, order="F")
    padded = np.concatenate([[0], pixels, [0]])
    changes = np.where(padded[1:] != padded[:-1])[0] + 1
    changes[1::2] -= changes[::2]
    return " ".join(map(str, changes))


wide = train_long.pivot(index="image_id", columns="defect_type", values="mask_rle").fillna("")
wide = wide.reindex(columns=CLASSES, fill_value="")
present = wide.applymap(lambda x: bool(str(x).strip()))
combo = present.astype(int).astype(str).agg("".join, axis=1)
combo_counts = combo.value_counts()
# Singleton combinations cannot be stratified; group only those as "rare".
strata = combo.where(combo.map(combo_counts) >= 2, "rare")
train_ids, val_ids = train_test_split(
    wide.index.to_numpy(), test_size=0.20, random_state=SEED, stratify=strata
)
print("split:", len(train_ids), len(val_ids))
print("combination counts:\n", combo.value_counts().sort_index())

In [ ]:
def find_image(folder, image_id):
    path = folder / f"{image_id}.jpg"
    if path.exists():
        return path
    matches = list(folder.glob(f"{image_id}.*"))
    if not matches:
        raise FileNotFoundError(image_id)
    return matches[0]


class SurfaceDataset(Dataset):
    def __init__(self, image_ids, masks=None, augment=False):
        self.ids = list(image_ids)
        self.masks = masks
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        folder = TRAIN_DIR if self.masks is not None else TEST_DIR
        image = cv2.imread(str(find_image(folder, image_id)), cv2.IMREAD_GRAYSCALE)
        assert image is not None and image.shape == (H, W), (image_id, None if image is None else image.shape)
        image = image.astype(np.float32) / 255.0
        image = (image - 0.5) / 0.25
        if self.masks is None:
            return torch.from_numpy(image[None]), image_id
        mask = np.stack([rle_decode(self.masks.loc[image_id, c]) for c in CLASSES]).astype(np.float32)
        if self.augment and random.random() < 0.5:
            image = np.ascontiguousarray(image[:, ::-1])
            mask = np.ascontiguousarray(mask[:, :, ::-1])
        return torch.from_numpy(image[None]), torch.from_numpy(mask), image_id


train_loader = DataLoader(SurfaceDataset(train_ids, wide, True), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(SurfaceDataset(val_ids, wide, False), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=2, pin_memory=True)

## Native-resolution compact U-Net baseline

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(),
        )
    def forward(self, x): return self.net(x)


class TinyUNet(nn.Module):
    def __init__(self, base=8):
        super().__init__()
        self.e1, self.e2, self.e3 = ConvBlock(1, base), ConvBlock(base, base*2), ConvBlock(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.mid = ConvBlock(base*4, base*8)
        self.u3 = nn.ConvTranspose2d(base*8, base*4, 2, 2)
        self.d3 = ConvBlock(base*8, base*4)
        self.u2 = nn.ConvTranspose2d(base*4, base*2, 2, 2)
        self.d2 = ConvBlock(base*4, base*2)
        self.u1 = nn.ConvTranspose2d(base*2, base, 2, 2)
        self.d1 = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, len(CLASSES), 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        x = self.mid(self.pool(e3))
        x = self.d3(torch.cat([self.u3(x), e3], 1))
        x = self.d2(torch.cat([self.u2(x), e2], 1))
        x = self.d1(torch.cat([self.u1(x), e1], 1))
        return self.out(x)


def loss_fn(logits, target):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    dims = (0, 2, 3)
    dice = (2 * (prob * target).sum(dims) + 1) / (prob.sum(dims) + target.sum(dims) + 1)
    return bce + (1 - dice.mean())


model = TinyUNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0
    for images, masks, _ in train_loader:
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            loss = loss_fn(model(images), masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        running += loss.item()
    mean_loss = running / len(train_loader)
    history.append({"epoch": epoch, "train_loss": mean_loss})
    print(f"epoch {epoch}/{EPOCHS} train_loss={mean_loss:.5f}")

torch.save(model.state_dict(), "/kaggle/working/milestone1_tiny_unet.pt")
pd.DataFrame(history).to_csv("/kaggle/working/training_history.csv", index=False)

## Holdout threshold sweep (no minimum-area filtering)

For each threshold I check: (a) how often an empty image/class pair gets a wrong mask,
(b) Dice on pairs that really have a defect, (c) the plain mean Dice over all pairs, and (d) the
competition style score, which also gives marks for getting empty masks right.

In [ ]:
def collect_validation():
    model.eval(); probs, truths = [], []
    with torch.no_grad():
        for images, masks, _ in val_loader:
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                p = torch.sigmoid(model(images.to(DEVICE, non_blocking=True)))
            probs.append(p.float().cpu().numpy().astype(np.float16))
            truths.append(masks.numpy().astype(np.uint8))
    return np.concatenate(probs), np.concatenate(truths)


val_prob, val_true = collect_validation()
true_positive = val_true.sum(axis=(2, 3)) > 0
empty_pair = ~true_positive
rows = []
for threshold in THRESHOLDS:
    pred = val_prob >= threshold
    pred_positive = pred.sum(axis=(2, 3)) > 0
    intersection = (pred & (val_true > 0)).sum(axis=(2, 3))
    denominator = pred.sum(axis=(2, 3)) + val_true.sum(axis=(2, 3))
    pair_dice = np.where(denominator > 0, 2 * intersection / np.maximum(denominator, 1), 1.0)
    fpr_empty = pred_positive[empty_pair].mean()
    fp_pixel_fraction = pred[empty_pair].mean()
    false_positive_score = 1.0 - fp_pixel_fraction
    empty_accuracy = 1.0 - fpr_empty
    foreground_dice = pair_dice[true_positive].mean()
    naive_dice = pair_dice.mean()
    # Official metric: 70% positive-pair Dice, 20% FP-pixel score, 10% empty accuracy.
    official_score = 0.70 * foreground_dice + 0.20 * false_positive_score + 0.10 * empty_accuracy
    rows.append(dict(threshold=threshold, empty_false_positive_rate=fpr_empty,
                     foreground_dice=foreground_dice, naive_mean_dice=naive_dice,
                     fp_pixel_fraction=fp_pixel_fraction,
                     false_positive_score=false_positive_score,
                     empty_accuracy=empty_accuracy, official_score=official_score))

results = pd.DataFrame(rows)
results.to_csv("/kaggle/working/threshold_sweep.csv", index=False)
print(results.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
best_official = float(results.loc[results.official_score.idxmax(), "threshold"])
best_naive = float(results.loc[results.naive_mean_dice.idxmax(), "threshold"])
print("best official threshold:", best_official, "best naive threshold:", best_naive)

first, last = results.iloc[0], results.iloc[-1]
q12 = (
    f"Across tau=0.3 to 0.9, the empty-pair false-positive rate changed from "
    f"{first.empty_false_positive_rate:.4f} to {last.empty_false_positive_rate:.4f}, while positive-pair "
    f"foreground Dice changed from {first.foreground_dice:.4f} to {last.foreground_dice:.4f}. "
    f"The official-style score peaked at tau={best_official:.1f}; naive mean Dice peaked at tau={best_naive:.1f}. "
    "Higher thresholds suppress spurious masks on the many empty image-class pairs, so the naive mean can "
    "prefer a higher tau because true-negative empty pairs dominate it. The official score gives 70% weight "
    "to foreground Dice, 20% to the false-positive pixel score, and 10% to empty accuracy, so it limits that "
    "empty-pair advantage and gives a fairer choice."
)
Path("/kaggle/working/milestone1_q12_answer.txt").write_text(q12)
print("\nQ12 answer:\n", q12)

del val_prob, val_true
gc.collect()
if DEVICE.type == "cuda": torch.cuda.empty_cache()

## Generate the real competition submission

In [ ]:
test_ids = sample.image_id.drop_duplicates().tolist()
test_loader = DataLoader(SurfaceDataset(test_ids), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=True)
prediction_by_key = {}
model.eval()
with torch.no_grad():
    for images, image_ids in test_loader:
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            probs = torch.sigmoid(model(images.to(DEVICE, non_blocking=True))).float().cpu().numpy()
        for i, image_id in enumerate(image_ids):
            for class_index, defect_type in enumerate(CLASSES):
                prediction_by_key[(image_id, defect_type)] = rle_encode(probs[i, class_index] >= best_official)

submission = sample.copy()
submission["mask_rle"] = [prediction_by_key[(i, c)] for i, c in zip(submission.image_id, submission.defect_type)]
# Kaggle's validator expects a literal space for an empty mask, matching sample_submission.csv.
submission["mask_rle"] = submission["mask_rle"].replace("", " ")
assert list(submission.columns) == ["ID", "image_id", "defect_type", "mask_rle"]
assert len(submission) == len(sample) and submission.ID.equals(sample.ID)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("submission rows:", len(submission), "non-empty masks:", (submission.mask_rle.str.len() > 0).sum())
print("Saved /kaggle/working/submission.csv")